# FRC Vision Scout — the engine: auto-label → train → **CPU-deployable** model (Colab T4)

Trains a **custom FUEL+robot detector** from this footage with **no manual labeling** (teacher
ensemble + SAHI), then exports an **OpenVINO INT8** model that runs **LIVE on your Intel Core
Ultra 7 (CPU / Arc iGPU / NPU) with no CUDA and no internet.** Drive-checkpointed so you can
stop/resume.

**Deploy target:** the slow SAHI teacher only labels frames here (once). The fast student learns
that recall and runs single-pass — so you get high recall *and* real-time on your own laptop.

**Run order:** `Runtime`→`T4 GPU`, then run cells top to bottom. Each stage saves to Drive; if a
session dies, re-run from the next stage. Paste any error back to Claude.

Stages: 1 setup · 2 upload clip · 3 auto-label (teacher+SAHI) · 4 train YOLO11s · 5 **export
OpenVINO INT8** + ONNX · 6 offline demo render · 7 package+download the deploy model. Then run
`src/live_openvino.py` on your machine for the live feed.

In [ ]:
#@title 1. Setup + GPU + Drive
# (no >/dev/null so install errors are visible; dropped deprecated openvino-dev)
!pip -q install ultralytics openvino nncf supervision sahi
import torch, os, ultralytics
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('ultralytics', ultralytics.__version__, '| GPU', torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount('/content/drive')
ROOT='/content/drive/MyDrive/frc_vision'; os.makedirs(ROOT, exist_ok=True)
DS=f'{ROOT}/dataset'; os.makedirs(DS, exist_ok=True)
print('workdir on Drive:', ROOT)

In [ ]:
#@title 2. Provide the match clip — UPLOAD (Colab can't download from YouTube)
# Colab's IP is bot-gated by YouTube. Put your clip on Drive at {ROOT}/match.mp4 (persists
# across sessions), OR just run this cell and upload it. Claude gives you match_720.mp4.
import os, shutil, cv2
MP4=f'{ROOT}/match.mp4'
if not (os.path.exists(MP4) and os.path.getsize(MP4)>0):
    from google.colab import files
    print('No match.mp4 on Drive — upload your clip now (e.g. match_720.mp4)...')
    up=files.upload()
    src=list(up.keys())[0]
    shutil.move(src, MP4)   # shutil.move (NOT os.replace) — works across Colab disk -> Drive
c=cv2.VideoCapture(MP4); n=int(c.get(7))
print('clip',int(c.get(3)),'x',int(c.get(4)),'@',round(c.get(5),1),'fps',n,'frames'); c.release()
assert n>0, 'No usable clip — upload match_720.mp4 (or copy it into the Drive frc_vision folder).'

In [ ]:
#@title 3. Auto-label (teacher ensemble: YOLO11x+SAHI for balls, YOLO-World for robots)
# Pseudo-labels, no human. 2 classes -> 0=ball, 1=robot. Writes YOLO dataset to Drive.
import cv2, glob, random, numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO
random.seed(0)
for s in ['images/train','images/val','labels/train','labels/val']: os.makedirs(f'{DS}/{s}',exist_ok=True)
ball_teacher=AutoDetectionModel.from_pretrained(model_type='ultralytics',model_path='yolo11x.pt',confidence_threshold=0.05,device='cuda:0')
world=YOLO('yolov8x-worldv2.pt'); world.set_classes(['robot'])
cap=cv2.VideoCapture(MP4); fps=cap.get(5); total=int(cap.get(7)); W=int(cap.get(3)); H=int(cap.get(4))
PLAY=(0.125,0.62)   # keep air column + carpet + front; drop scoreboard/reflection
step=max(1,int(fps*0.5))   # ~2 frames/sec sampled for labeling
idx=n=0
while True:
    ok=cap.grab()
    if not ok: break
    if idx%step==0:
        ok2,f=cap.retrieve()
        if not ok2: break
        cv2.imwrite('f.png',f)
        sp=get_sliced_prediction('f.png',ball_teacher,slice_height=384,slice_width=384,overlap_height_ratio=0.25,overlap_width_ratio=0.25,verbose=0)
        lines=[]
        for o in sp.object_prediction_list:
            if o.category.id!=32: continue
            b=o.bbox; cx=(b.minx+b.maxx)/2; cy=(b.miny+b.maxy)/2
            if not (PLAY[0]*H<=cy<=PLAY[1]*H): continue
            lines.append(f'0 {cx/W:.6f} {cy/H:.6f} {(b.maxx-b.minx)/W:.6f} {(b.maxy-b.miny)/H:.6f}')
        for box in world.predict(f,conf=0.05,imgsz=1280,verbose=False)[0].boxes.xyxy.cpu().numpy():
            x1,y1,x2,y2=box; cy=(y1+y2)/2
            if not (PLAY[0]*H<=cy<=PLAY[1]*H): continue
            lines.append(f'1 {((x1+x2)/2)/W:.6f} {cy/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
        sub='val' if random.random()<0.15 else 'train'
        cv2.imwrite(f'{DS}/images/{sub}/{n:05d}.png',f)
        open(f'{DS}/labels/{sub}/{n:05d}.txt','w').write(chr(10).join(lines))
        n+=1
        if n%20==0: print('labeled',n,'frames')
    idx+=1
cap.release()
open(f'{DS}/data.yaml','w').write(f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n  1: robot\n")
print('AUTO-LABEL done:',n,'frames ->',DS)

In [ ]:
#@title 4. Train the specialist student (YOLO11s) — AUTO-RESUMES if disconnected
# YOLO11s = sweet spot for Intel iGPU/NPU live inference. For max CPU speed swap 'yolo11n.pt'.
# Checkpoints save to Drive every epoch -> if Colab drops, just RE-RUN this cell and it resumes.
from ultralytics import YOLO
import os
ckpt=f'{ROOT}/runs/fuel_robot/weights/last.pt'
if os.path.exists(ckpt):
    print('Found checkpoint -> RESUMING from', ckpt)
    m=YOLO(ckpt); m.train(resume=True)
else:
    print('Fresh training run')
    m=YOLO('yolo11s.pt')
    m.train(data=f'{DS}/data.yaml', epochs=120, imgsz=1024, batch=16, device=0,
            project=f'{ROOT}/runs', name='fuel_robot', exist_ok=True,
            mosaic=1.0, close_mosaic=15, copy_paste=0.3, hsv_v=0.5, hsv_s=0.5,
            degrees=5, translate=0.1, scale=0.5, fliplr=0.5, erasing=0.3,
            patience=40, cache=False)
BEST=f'{ROOT}/runs/fuel_robot/weights/best.pt'
print('trained ->',BEST)

In [ ]:
#@title 5. Export OpenVINO INT8 (CUDA-free deploy artifact) + ONNX
# This is what runs LIVE on your Intel Core Ultra (CPU/iGPU/NPU). INT8 is calibrated on YOUR
# dataset -> small + fast with minimal accuracy loss. DEPLOY_IMGSZ fixes the live input size.
from ultralytics import YOLO
import glob
BEST=f'{ROOT}/runs/fuel_robot/weights/best.pt'
DEPLOY_IMGSZ=768          # 640 = faster, 960 = higher small-ball recall. Must match live script.
m=YOLO(BEST)
ov=m.export(format='openvino', int8=True, data=f'{DS}/data.yaml', imgsz=DEPLOY_IMGSZ, half=False)
print('OpenVINO INT8 ->', ov)
onnx=m.export(format='onnx', imgsz=DEPLOY_IMGSZ, opset=13, simplify=True)   # portable fallback
print('ONNX ->', onnx)
# (optional) TensorRT for a Colab-T4 backup live path:
try:
    eng=m.export(format='engine', half=True, imgsz=DEPLOY_IMGSZ, device=0); print('TensorRT ->',eng)
except Exception as e: print('TensorRT skipped:', str(e)[:90])
# quick sanity predict through the INT8 model on a dataset image (no dependency on cell 2's MP4)
imgs=glob.glob(f'{DS}/images/val/*.png') or glob.glob(f'{DS}/images/train/*.png')
if imgs:
    r=YOLO(ov, task='detect').predict(imgs[0], imgsz=DEPLOY_IMGSZ, conf=0.15, verbose=False)[0]
    print('INT8 model OK — detections on a sample frame:', len(r.boxes))

In [ ]:
#@title 6. (Optional) Track the full match -> annotated demo video
from ultralytics import YOLO
import cv2, numpy as np, json, os
MP4=f'{ROOT}/match.mp4'   # self-defines so this works even if cell 2 was skipped after a reconnect
BEST=f'{ROOT}/runs/fuel_robot/weights/best.pt'
m=YOLO(BEST)
res=m.track(source=MP4, imgsz=1280, conf=0.15, iou=0.5, tracker='botsort.yaml', persist=True, stream=True, verbose=False)
cap=cv2.VideoCapture(MP4); fps=cap.get(5); W=int(cap.get(3)); H=int(cap.get(4)); cap.release()
vw=cv2.VideoWriter('out.mp4',cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
COL={0:(0,255,255),1:(255,90,0)}; per_frame=[]
for r in res:
    f=r.orig_img.copy(); nb=nr=0
    if r.boxes is not None and len(r.boxes):
        ids=r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else [-1]*len(r.boxes)
        for b,c,tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), ids):
            x1,y1,x2,y2=b.astype(int)
            if c==0: cv2.circle(f,((x1+x2)//2,(y1+y2)//2),max(3,(x2-x1)//2),COL[0],2); nb+=1
            else: cv2.rectangle(f,(x1,y1),(x2,y2),COL[1],2); cv2.putText(f,f'R{tid}',(x1,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.5,COL[1],1); nr+=1
    cv2.putText(f,f'FUEL {nb}  ROBOTS {nr}',(8,28),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
    vw.write(f); per_frame.append({'balls':nb,'robots':nr})
vw.release()
!ffmpeg -hide_banner -loglevel error -y -i out.mp4 -c:v libx264 -pix_fmt yuv420p {ROOT}/tracked_custom.mp4
json.dump(per_frame,open(f'{ROOT}/tracks.json','w'))
print('mean balls/frame',round(np.mean([p['balls'] for p in per_frame]),1),'mean robots/frame',round(np.mean([p['robots'] for p in per_frame]),1))
print('saved',f'{ROOT}/tracked_custom.mp4')

In [ ]:
#@title 7. Package + download the DEPLOY model (OpenVINO) + demo video + mAP
import shutil, os, glob
from google.colab import files
from ultralytics import YOLO
run=f'{ROOT}/runs/fuel_robot'; wdir=f'{run}/weights'
ovdirs=[d for d in os.listdir(wdir) if d.endswith('openvino_model')]
assert ovdirs, 'No OpenVINO export found — run cell 5 first.'
zip_base=f'{ROOT}/frc_model_openvino'
shutil.make_archive(zip_base, 'zip', f'{wdir}/{ovdirs[0]}')
print('packaged', ovdirs[0], '->', zip_base+'.zip')
try: print('val mAP50-95:', YOLO(f'{wdir}/best.pt').val(data=f'{ROOT}/dataset/data.yaml').box.map)
except Exception as e: print('val skipped:', str(e)[:80])
files.download(zip_base+'.zip')
if os.path.exists(f'{ROOT}/tracked_custom.mp4'): files.download(f'{ROOT}/tracked_custom.mp4')
print('\nNEXT: unzip frc_model_openvino.zip on your laptop, then:')
print('  python src/live_openvino.py <unzipped_dir> intel:gpu data/match_1080p.mp4')